# RL-Augmented Vision Web Agent — full Colab pipeline

Runs the whole thing on a Colab A100. PPO checkpoints land on Drive so auto-resume survives disconnects.

**Colab secrets (key icon in sidebar):**
- `GEMINI_API_KEY` (required)
- `WANDB_API_KEY` (optional)
- `GH_USER`, `GH_REPO` (only if cloning from GitHub)

**Runtime:** GPU → A100 80GB; High-RAM.

In [ ]:
# 1. Verify A100
!nvidia-smi | head -20

In [ ]:
# 2. Mount Drive + load secrets
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/rlwa'
os.makedirs(f'{DRIVE}/hf', exist_ok=True)
os.makedirs(f'{DRIVE}/ckpts', exist_ok=True)
os.makedirs(f'{DRIVE}/traces', exist_ok=True)
os.makedirs(f'{DRIVE}/eval', exist_ok=True)
os.makedirs(f'{DRIVE}/traj', exist_ok=True)
os.environ['HF_HOME']        = f'{DRIVE}/hf'
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
try:    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
except: os.environ['WANDB_MODE']    = 'offline'

In [ ]:
# 3. System deps + virtual display for headless Chromium
!apt-get -qq install -y xvfb libnss3 libgbm1 libasound2 libatk-bridge2.0-0 libxkbcommon0 libxrandr2 libxcomposite1 libxdamage1
import subprocess, os
subprocess.Popen(['Xvfb', ':99', '-screen', '0', '1280x720x24'])
os.environ['DISPLAY'] = ':99'

In [ ]:
# 4. Get the code: either clone from GitHub OR upload the project zip and unzip it.
# Option A — clone:
from google.colab import userdata
try:
    GH_USER = userdata.get('GH_USER'); GH_REPO = userdata.get('GH_REPO')
    if not os.path.exists('/content/rlwa'):
        !git clone https://github.com/{GH_USER}/{GH_REPO} /content/rlwa
except Exception:
    print('No GH_* secrets — upload project.zip via the Files pane, then:')
    print('  !unzip -q /content/project.zip -d /content/rlwa')
%cd /content/rlwa

In [ ]:
# 5. Install package + Playwright
!pip install -q -e . gradio pytest
!playwright install chromium

In [ ]:
# 6. Smoke tests (engineering signal — runs in ~5s, no API calls)
!pytest -q

In [ ]:
# 7. Pre-cache frozen encoders to Drive (skips re-download next session)
from huggingface_hub import snapshot_download
snapshot_download('google/siglip-base-patch16-224',  cache_dir=os.environ['HF_HOME'])
snapshot_download('sentence-transformers/all-MiniLM-L6-v2', cache_dir=os.environ['HF_HOME'])

In [ ]:
# 8. End-to-end smoke test (1 episode through Gemini)
!python scripts/00_smoke_test.py --task miniwob.click-button --seed 0

## Phase 1 — baselines & BC pretraining

In [ ]:
# 9. Prompt-only quick baseline (3 seeds, sanity check)
!python scripts/04_eval.py --agent prompt_only --seeds 3 --workers 32 --planner-workers 64

In [ ]:
# 10. Collect BC traces (~30-60 min with async Gemini)
!python scripts/01_collect_bc_traces.py --episodes-per-task 100 --out data/traces/bc.jsonl --workers 32 --planner-workers 96

In [ ]:
# 11. BC pretrain (~10 min on A100)
!python scripts/02_train_bc.py

## Phase 2 — PPO (auto-resume from Drive)

If Colab disconnects mid-training, just re-run this cell — `PPOTrainer.__init__` reloads `ppo_last.pt` (policy + optimizer + global_step) from the save_dir.

In [ ]:
# 12. PPO (80k env-steps, ~1.5-2h with N=32 envs + async planner).
# save_dir points at Drive so auto-resume survives a disconnect.
!python scripts/03_train_ppo.py \
    --use-wandb False \
    --planner-workers 48 \
    --save-dir /content/drive/MyDrive/rlwa/ckpts
# After it finishes, copy the best checkpoint to the local location every script expects:
!mkdir -p data/checkpoints && cp /content/drive/MyDrive/rlwa/ckpts/ppo_best.pt data/checkpoints/ppo_best.pt
!cp /content/drive/MyDrive/rlwa/ckpts/ppo_log.jsonl data/checkpoints/ppo_log.jsonl || true

## Phase 3 — full evaluation suite (all baselines + RL)

In [ ]:
# 13. Main results table — 5 agents × 5 seeds × 8 tasks
!python scripts/04_eval.py --agent random       --seeds 5 --workers 32 --planner-workers 64
!python scripts/04_eval.py --agent prompt_only  --seeds 5 --workers 32 --planner-workers 64
!python scripts/04_eval.py --agent reflective   --seeds 5 --workers 32 --planner-workers 64
!python scripts/04_eval.py --agent bc_only      --seeds 5 --workers 32 --planner-workers 64
!python scripts/04_eval.py --agent rl_refiner   --seeds 5 --workers 32 --planner-workers 64

In [ ]:
# 14. Robustness (CSS-jitter perturbation)
!python scripts/05_perturb_eval.py --agent prompt_only --seeds 3 --workers 16 --planner-workers 64
!python scripts/05_perturb_eval.py --agent rl_refiner  --seeds 3 --workers 16 --planner-workers 64

In [ ]:
# 15. Ablations (5 variants × 40k steps each) — leave this overnight if tight on time
!python scripts/06_ablations.py --planner-workers 48

In [ ]:
# 16. Top-K oracle headroom
!python scripts/07_oracle_topk.py --seeds 3 --workers 32 --planner-workers 64

## Phase 4 — figures, trajectories, report

In [ ]:
# 17. All plots (per-task SR, robustness, learning curve, ablations)
!python scripts/08_plots.py

In [ ]:
# 18. Record side-by-side trajectory HTMLs (prompt_only vs rl_refiner) for viva
!python scripts/09_record_trajectories.py --seed 0 --out-dir data/traj

In [ ]:
# 19. Auto-fill numbers into the report from all *_summary.json files
!python scripts/10_fill_report.py
from IPython.display import Markdown, display
display(Markdown(open('paper/report.md').read()))

In [ ]:
# 20. Persist everything to Drive
!cp -r data/eval data/checkpoints data/traces data/traj paper/figures paper/report.md /content/drive/MyDrive/rlwa/ 2>/dev/null || true
!ls /content/drive/MyDrive/rlwa/